### Importing Libraries

In [3]:
# Run this cell ONLY ONCE to set the project root as the working directory. 
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\yeshw\Documents\Gayathri personal doc\Gayathri\arizona\INFO 556 - Text Retrieval and Web Search\project\hm-fashion-recommendation


In [4]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import pandas as pd
import pickle
from config import ROOT_DIR
from src.models.baseline_model import create_baseline_recommender,create_recency_baseline
from src.models.als_model import train_and_save_als_model, create_als_recommender
from src.evaluation.metrics import evaluate_all, evaluate_candidates
from src.candidate_retrieval.PreviousPurchases import PreviousPurchases
from src.candidate_retrieval.ItemSimilarity import ItemSimilarityCandidates
from src.candidate_retrieval.RecencyCandidates import RecencyCandidates
from src.candidate_retrieval.AlsCandidates import ALSCandidates
from src.candidate_retrieval.StyleAlternatives import SameProductColorCandidates
from src.candidate_retrieval.CandidateRetrievalPipeline import CandidateGeneratorEnsemble
from src.utils.generate_labels import create_labels
from src.feature_engineer.feature_generator import FeatureGenerator
from src.utils.downsample_candidates import downsample_negatives
from src.models.rank_model import train_ranker, create_lgbm_recommender
from pathlib import Path

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
cleaned_df_folder = "input/cleaned"
def load_from_parquet(cleaned_df_name):
    print("Loading data from Parquet...")
    return pd.read_parquet(cleaned_df_folder + "/" + cleaned_df_name + ".parquet")

In [6]:
cols_needed = ['t_dat', 'customer_id', 'article_id','week']

train_transactions = pd.read_parquet(
    "input/cleaned/train_transactions.parquet", 
    columns=cols_needed
)

In [7]:
test_transactions = load_from_parquet("test_transactions")
val_transactions = load_from_parquet("val_transactions")
articles = load_from_parquet("articles")
customers = load_from_parquet("customers")

Loading data from Parquet...
Loading data from Parquet...
Loading data from Parquet...
Loading data from Parquet...


In [8]:
# week 2 is the latest week in the training data, so we will use it as the target week for training the ranker model. 
# The transactions from week 2 will be used to create the labels for our training data, while the transactions from weeks 3 to 104 will be used as the historical data for generating candidates.

train_target_week = train_transactions['week'].min() # week 2
train_target_tx = train_transactions[train_transactions['week'] == train_target_week] # week 2
train_history_tx = train_transactions[train_transactions['week'] > train_target_week] # Weeks 3 to 104

In [ ]:
#Path to save the trained ALS model and ranker model
als_train_history_file_path = ROOT_DIR / 'saved_models' / 'als_recommender_model_training_history.pkl'
als_val_history_file_path = ROOT_DIR / 'saved_models' / 'als_recommender_model_val_history.pkl'
als_test_history_file_path = ROOT_DIR / 'saved_models' / 'als_recommender_model_test_history.pkl'

ranker_model_file_path = ROOT_DIR / 'saved_models' / 'ranker_model.pkl'

## Final Pipeline and Candidate Generation

### Train target candidate Generation

In [ ]:
# Train als model for train_history
train_and_save_als_model(
    train=train_history_tx,
    model_path=als_train_history_file_path,
    lookback_weeks=14,
    factors=100,
    iterations=15,
    regularization=0.01,
    alpha=40.0,
    random_seed=42
)

In [24]:
train_target_customers = train_target_tx['customer_id'].unique()

generators_config = {
    "previous_purchases": (PreviousPurchases, {"lookback_weeks": 12, "k": 15}),
    "item_similarity": (
        ItemSimilarityCandidates,
        {
            "lookback_weeks": 6,
            "min_item_frequency": 30,
            "k": 25
        },
    ),
    "recency": (RecencyCandidates, {"lookback_weeks": 12, "k": 20}),
    "als": (ALSCandidates, {"model_path": als_train_history_file_path, "k": 40}),
    "same_product_color": (SameProductColorCandidates, {"lookback_weeks": 12, "k": 20})
}
ensemble = CandidateGeneratorEnsemble(
    transactions=train_history_tx, articles=articles, generators=generators_config
)

# Generate candidates for train target customers
print(
    f"Generating candidates via ensemble for {len(train_target_customers):,} train target"
    " customers..."
)
train_target_recs_df = ensemble.generate(users=train_target_customers)

print(f"Generated {len(train_target_recs_df):,} total training candidate rows.")
evaluate_candidates(train_target_recs_df, train_target_tx, k=120)
train_target_recs_df.to_parquet(cleaned_df_folder + "/candidates_train.parquet", index=False)

Generating candidates via ensemble for 75,822 train target customers...
Generated previous purchase candidates
Generated candidate scores for users
Generated item similarity candidates
Generated recency candidates
Generated als candidates
Generated style alternatives candidates
Generated 5,554,999 total training candidate rows.
Preparing evaluation matrices...
Recall@120: 0.1184


In [26]:
train_target_recs_df.head()

,customer_id,article_id,source
2853207,ffffcf35913a0bee60e8741cb2b4e78b8a98ee5ff2e6a1...,0915529001,recency
442051,ffffcf35913a0bee60e8741cb2b4e78b8a98ee5ff2e6a1...,0816759002,previous_purchases
442049,ffffcf35913a0bee60e8741cb2b4e78b8a98ee5ff2e6a1...,0835561003,previous_purchases
442048,ffffcf35913a0bee60e8741cb2b4e78b8a98ee5ff2e6a1...,0636323002,previous_purchases
2853211,ffffcf35913a0bee60e8741cb2b4e78b8a98ee5ff2e6a1...,0852584001,recency


In [27]:

# Create labels using the training target week transactions
print("Generating labels for candidates")
train_labelled = create_labels(train_target_recs_df, train_target_tx)
# Extract features using the historical transactions
print("Adding features to the candidates")
generator = FeatureGenerator(
    train_transactions_df=train_history_tx,
    articles_df=articles,
    customers_df=customers
)
train_candidate_features = generator.transform(train_labelled)
train_candidate_features.to_parquet(cleaned_df_folder + "/train_candidates_features.parquet", index=False)
# 6. Downsample negatives for training
print("Downsampling candidates")
train_downsampled = downsample_negatives(train_candidate_features, max_negatives_per_user=30)
train_downsampled.to_parquet(cleaned_df_folder + "/train_candidates_downsampled.parquet", index=False)

Generating labels for candidates
Extracting unique purchase pairs from target set...
Found 238,074 unique customer-article purchase pairs in target week.
Total candidates before labeling: 5,554,999
Positive samples captured in candidate pool: 23,737 (0.43%)
Adding features to the candidates
-> Adding item metadata and multi-window popularities...
-> Adding user demographics and profile features...
-> Adding user-item interaction affinity features...
-> Processing candidate retrieval flags and scores...
Feature matrix successfully built. Final shape: (5554999, 28)
Downsampling candidates
Original dataset shape: (5554999, 28)
Original positive count: 23,737
Sampling up to 30 negatives per user...
Downsampled dataset shape: (2110981, 28)
New positive proportion: 0.0112 (1.12%)


### Train Ranker model with train_target candidates

In [17]:
train_downsampled = load_from_parquet("train_candidates_downsampled")

Loading data from Parquet...


In [18]:
# Prepare training structures for LightGBM
train_df = train_downsampled.sort_values("customer_id").reset_index(drop=True)
train_baskets = train_df.groupby("customer_id", sort=False).size().to_numpy()

drop_features = ["customer_id", "article_id", "target"]
feature_cols = [col for col in train_df.columns if col not in drop_features]

X_train = train_df[feature_cols].copy()
y_train = train_df["target"]

# Convert categorical columns
categorical_cols = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

for col in categorical_cols:
    X_train[col] = X_train[col].astype("category")
    
# Save categorical mapping categories for runtime alignment
train_feature_categories = {
    col: X_train[col].cat.categories 
    for col in categorical_cols
}

In [ ]:
print("Training LightGBM ranker model...")
model, categorical_cols = train_ranker(
    X_train,
    y_train,
    train_baskets,
    boosting_type="dart",
    n_estimators=200,
    min_child_samples=75,
    learning_rate=0.05
)

with open(ranker_model_file_path, 'wb') as f:
    pickle.dump(model, f)

Training LightGBM ranker model...


### Validation Candidates Generation

In [ ]:
# Train als model for val_history
train_and_save_als_model(
    train=train_transactions,
    model_path=als_val_history_file_path,
    lookback_weeks=14,
    factors=100,
    iterations=15,
    regularization=0.01,
    alpha=40.0,
    random_seed=42
)

In [ ]:
val_customers = val_transactions['customer_id'].unique()

generators_config = {
    "previous_purchases": (PreviousPurchases, {"lookback_weeks": 12, "k": 15}),
    "item_similarity": (
        ItemSimilarityCandidates,
        {
            "lookback_weeks": 6,
            "min_item_frequency": 30,
            "k": 25
        },
    ),
    "recency": (RecencyCandidates, {"lookback_weeks": 12, "k": 20}),
    "als": (ALSCandidates, {"model_path": als_val_history_file_path, "k": 40}),
    "same_product_color": (SameProductColorCandidates, {"lookback_weeks": 12, "k": 20})
}

# Initialize the Ensemble
ensemble = CandidateGeneratorEnsemble(
    transactions=train_transactions, articles=articles, generators=generators_config
)

# Generate candidates for validation customers
print(
    f"Generating candidates via ensemble for {len(val_customers):,} validation"
    " customers..."
)
recs_df = ensemble.generate(users=val_customers)

print(f"Generated {len(recs_df):,} total candidate rows.")
evaluate_candidates(recs_df, val_transactions, k=120)
recs_df.to_parquet(cleaned_df_folder + "/candidates_val.parquet", index=False)

Generating candidates via ensemble for 72,019 validation customers...
Generated previous purchase candidates
Generated candidate scores for users
Generated item similarity candidates
Generated recency candidates
Generated als candidates
Generated style alternatives candidates
Generated 4,937,524 total candidate rows.
Preparing evaluation matrices...
Recall@120: 0.1126


0.11260310714889189

In [10]:
# Generate features for the Candidates
generator = FeatureGenerator(
    train_transactions_df=train_transactions,
    articles_df=articles,
    customers_df=customers
)

val_candidate_features_df = generator.transform(recs_df)
val_candidate_features_df.to_parquet(cleaned_df_folder + "/val_candidate_features.parquet", index=False)


-> Adding item metadata and multi-window popularities...
-> Adding user demographics and profile features...
-> Adding user-item interaction affinity features...
-> Processing candidate retrieval flags and scores...
Feature matrix successfully built. Final shape: (4937524, 27)


In [16]:
from src.models.rank_model import create_lgbm_recommender
print("Scoring candidates with LightGBM ranker...")
lgbm_recommender_func = create_lgbm_recommender(
    ranker=model,
    candidate_features=val_candidate_features_df,
    categorical_cols=categorical_cols,
    train_feature_categories=train_feature_categories
)
# Generate top 12 recommendations for validation customers
# val_predictions = lgbm_recommender_func(val_customers.tolist(), k=12)

# Prepare ground-truth actual purchases for Week 0
val_gt = val_transactions[['customer_id', 'article_id']]

# Evaluate validation score!
validation_map = evaluate_all(lgbm_recommender_func, val_gt, k=12)

Scoring candidates with LightGBM ranker...
Generating predictions for 72,019 validation customers...
Preparing evaluation matrices...
Computing metrics...


In [ ]:

print(f"Validation MAP@12 Score: {validation_map['MAP@12']:.5f}")
print("All Validation Metrics:", validation_map)

Validation MAP@12 Score: 0.02702
All Validation Metrics: {'MAP@12': 0.027023405642728395, 'HitRate@12': 0.11901026118107721, 'NDCG@12': 0.040762508862400425, 'Recall@12': 0.05852373646558842}


## Prediction on Test Dataset

In [10]:

# Combine  training and validation transactions 
train_val_combined = pd.concat([train_transactions, val_transactions], ignore_index=True)


In [11]:
# Train als model for test_history
train_and_save_als_model(
    train=train_val_combined,
    model_path=als_test_history_file_path,
    lookback_weeks=14,
    factors=100,
    iterations=15,
    regularization=0.01,
    alpha=40.0,
    random_seed=42
)

1. User map shape: 558942
2. Item map shape: 43892
3. Reverse item map shape: 43892


c:\Users\yeshw\AppData\Local\Programs\Python\Python312\Lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 4 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


Training Implicit ALS model...


  0%|          | 0/15 [00:00<?, ?it/s]

Training complete!
ALS model saved to C:\Users\yeshw\Documents\Gayathri personal doc\Gayathri\arizona\INFO 556 - Text Retrieval and Web Search\project\hm-fashion-recommendation\saved_models\als_recommender_model_test_history.pkl


{'model': <implicit.cpu.als.AlternatingLeastSquares at 0x2d70ca81820>,
 'user_map': {'0009344276939620e81b41ab8958edca7c2a5b9d6ee45dcd599161d480b9a5d3': 0,
  '000b8ae9083661ded5f13a83a214b2de88fd3ce55bee5eb7d4b69ec60051ef75': 1,
  '000fa1b80857fa40bf25990bc1b1b65afc63923a8e4b5762db34e4bb46032d29': 2,
  '001143bec624de9df82bd687babb382e00b0285e2371cc9120de5d42c48f1076': 3,
  '001474a6235ebaa675ef2313284c3a9c7490f6972af132943764ee76bbc174d6': 4,
  '001f5299820c00df306221ff581abf9d18507c2e35ecb34aceed093612ac84d3': 5,
  '0028705bb2be02776dd451e321d033ab791ceae6e5d3fd652e101782f62b3750': 6,
  '002cc37730b9d96472466a5f6f3eca5fddcd1918fc1c6c09982ee26d2b7be806': 7,
  '00308902655338899a696e9bea874123cbcd012130e65f3ee57d2bd07425f0b1': 8,
  '0035b1a78ff4c2ae30b91308700c09d257bc8f830acc53574829f391f3e611d4': 9,
  '00368437741becbed496afd6239c8fc5c9d6bf98600a52f498a8cab0cd4af01f': 10,
  '003766bbac747bf7f9603b778f3d523d3cea506b3e5239f2e8a9673b471761a6': 11,
  '0038bf2b66fdc1de4e25bce86c50d6dbdc3a

In [12]:
test_customers = test_transactions['customer_id'].unique()

generators_config = {
    "previous_purchases": (PreviousPurchases, {"lookback_weeks": 12, "k": 15}),
    "item_similarity": (
        ItemSimilarityCandidates,
        {
            "lookback_weeks": 6,
            "min_item_frequency": 30,
            "k": 25
        },
    ),
    "recency": (RecencyCandidates, {"lookback_weeks": 12, "k": 20}),
    "als": (ALSCandidates, {"model_path": als_test_history_file_path, "k": 40}),
    "same_product_color": (SameProductColorCandidates, {"lookback_weeks": 12, "k": 20})
}

full_ensemble = CandidateGeneratorEnsemble(
    transactions=train_val_combined,  # data (train + validation combined)
    articles=articles, 
    generators=generators_config
)

# Generate candidates for test customers
print(f"Generating test candidates for {len(test_customers):,} test customers...")
test_recs_df = full_ensemble.generate(users=test_customers)
test_recs_df.to_parquet(cleaned_df_folder + "/candidates_test.parquet", index=False)

Generating test candidates for 68,984 test customers...
Generated previous purchase candidates
Generated candidate scores for users
Generated item similarity candidates
Generated recency candidates
Generated als candidates
Generated style alternatives candidates


In [ ]:
from src.feature_engineer.feature_generator import FeatureGenerator
# Extract features for test candidates
print("Extracting features for test candidates...")
generator = FeatureGenerator(
    train_transactions_df=train_val_combined,
    articles_df=articles,
    customers_df=customers
)

test_candidate_features_df = generator.transform(test_recs_df)
test_candidate_features_df.to_parquet(cleaned_df_folder + "/test_candidate_features.parquet", index=False)

Extracting features for test candidates...
-> Adding item metadata and multi-window popularities...
-> Adding user demographics and profile features...
-> Adding user-item interaction affinity features...
-> Processing candidate retrieval flags and scores...
Feature matrix successfully built. Final shape: (4960252, 27)


In [22]:

# Score and rank them using trained ranker
test_recommender_func = create_lgbm_recommender(
    ranker=model,
    candidate_features=test_candidate_features_df,
    categorical_cols=categorical_cols,
    train_feature_categories=train_feature_categories
)


# Prepare ground-truth actual purchases for Week 0
val_gt = test_transactions[['customer_id', 'article_id']]

# Evaluate validation score!
validation_map = evaluate_all(test_recommender_func, val_gt, k=12)

print(f"Validation MAP@12 Score: {validation_map['MAP@12']:.5f}")
print("All Validation Metrics:", validation_map)

Generating predictions for 68,984 validation customers...
Preparing evaluation matrices...
Computing metrics...
Validation MAP@12 Score: 0.02814
All Validation Metrics: {'MAP@12': 0.028144874860847033, 'HitRate@12': 0.1235069001507596, 'NDCG@12': 0.04260403347297303, 'Recall@12': 0.06322636538072861}
